<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/4%20-%20Redes%20Neurais/4.6_mnist_pipeline_completo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 4.6 — Pipeline completo de treinamento no MNIST

O problema é o mesmo dos notebooks 3.2 e 3.4: reconhecer o dígito (0 a 9) escrito à mão numa imagem 28×28 em tons de cinza do MNIST. A diferença é que agora juntamos, num só programa, tudo o que foi visto até aqui: `Dataset` e `DataLoader` com mini-batches, uma rede com várias camadas, dropout, um otimizador adaptativo, ajuste da taxa de aprendizado ao longo das épocas, avaliação num conjunto de teste separado, execução em GPU quando houver, e gravação do modelo treinado.

O laço de treino, o otimizador e o agendador seguem o exemplo oficial de MNIST do PyTorch (https://github.com/pytorch/examples/tree/main/mnist). A rede do exemplo oficial é convolucional; aqui ela foi trocada por uma rede densa, com as camadas `nn.Linear` vistas no módulo 4.

## Objetivos

- Montar o pipeline completo: dados → modelo → função de custo → otimizador → laço de treino → avaliação no teste → gravação do modelo.
- Usar `device` para treinar na GPU quando disponível, e entender `model.train()` × `model.eval()` e `torch.no_grad()`.
- Treinar uma rede densa com duas camadas escondidas e dropout, contando os parâmetros de cada camada.
- Usar um agendador da taxa de aprendizado (`StepLR`) e salvar/carregar os parâmetros com `state_dict`.

## Importação das bibliotecas

In [ ]:
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torchvision import datasets, transforms

torch.manual_seed(1111)

# device: onde ficam os tensores e onde as contas são feitas ("cuda" = GPU NVIDIA, "cpu" = processador)
device = "cuda" if torch.cuda.is_available() else "cpu"
print('device:', device)

## Dados: MNIST

`datasets.MNIST` baixa o conjunto (60.000 imagens de treino e 10.000 de teste) para a pasta `./data` na primeira execução. Cada imagem passa por duas transformações:

- `transforms.ToTensor()`: converte a imagem num tensor de shape (1, 28, 28) — 1 canal (tons de cinza), 28 linhas, 28 colunas — com valores entre 0 e 1;
- `transforms.Normalize((0.1307,), (0.3081,))`: subtrai a média (0,1307) e divide pelo desvio-padrão (0,3081) dos pixels do conjunto de treino, deixando os valores com média ≈ 0 e desvio ≈ 1.

O treino usa mini-batches de 64 imagens, embaralhados a cada época (`shuffle=True`). O teste usa lotes de 1000 e não precisa ser embaralhado: ele só é avaliado, nunca usado para atualizar os parâmetros. Na GPU, `num_workers` e `pin_memory` aceleram a cópia dos dados para a placa.

In [ ]:
train_kwargs = {'batch_size': 64, 'shuffle': True}
test_kwargs = {'batch_size': 1000}
if device == "cuda":
    cuda_kwargs = {'num_workers': 1, 'pin_memory': True}
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
    ])
dataset1 = datasets.MNIST('./data', train=True, download=True, transform=transform)
dataset2 = datasets.MNIST('./data', train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(dataset1, **train_kwargs)
test_loader = torch.utils.data.DataLoader(dataset2, **test_kwargs)

data, target = next(iter(train_loader))   # um mini-batch de treino
print('treino:', len(dataset1), 'imagens,', len(train_loader), 'mini-batches por época')
print('teste: ', len(dataset2), 'imagens,', len(test_loader), 'lotes')
print('mini-batch: data', tuple(data.shape), ' target', tuple(target.shape))

In [ ]:
fig, axs = plt.subplots(1, 10, figsize=(15, 2))
for ax, img, label in zip(axs, data, target):
    ax.imshow(img[0], cmap='gray')
    ax.set_title(f'rótulo: {label.item()}')
    ax.axis('off')
plt.show()

## Modelo

A rede é densa, como as dos notebooks 4.1–4.3, agora com entrada de imagem:

- `torch.flatten(x, 1)`: achata cada imagem `(1, 28, 28)` num vetor de 784 pixels, mantendo a dimensão 0 (o mini-batch). É o mesmo que o `x.view(-1, 784)` dos notebooks 3.2 e 3.4.
- `fc1` (784 → 512) e `fc2` (512 → 256) são as camadas escondidas, com ReLU; `fc3` (256 → 10) produz os 10 logits, um por dígito.
- `nn.Dropout(p)`: durante o treino, zera aleatoriamente uma fração `p` das ativações (e multiplica as restantes por $1/(1-p)$); na avaliação não faz nada. É uma forma de regularização, como as do notebook 4.3: a rede não pode depender de um neurônio específico. Aqui usamos `p = 0,2` depois de cada camada escondida.
- `F.log_softmax(x, dim=1)`: aplica o logaritmo do softmax aos 10 logits de cada amostra. A saída são log-probabilidades das 10 classes.

Ao achatar a imagem, a rede perde a informação de quais pixels são vizinhos: para ela, a entrada é uma lista de 784 números. As redes convolucionais, que aproveitam essa vizinhança, são assunto de um módulo próprio.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)
        self.dropout1 = nn.Dropout(0.2)
        self.dropout2 = nn.Dropout(0.2)

    def forward(self, x):          # x: (N, 1, 28, 28)
        x = torch.flatten(x, 1)    # (N, 784)
        x = self.fc1(x)            # (N, 512)
        x = F.relu(x)
        x = self.dropout1(x)
        x = self.fc2(x)            # (N, 256)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc3(x)            # (N, 10): logits
        output = F.log_softmax(x, dim=1)   # (N, 10): log-probabilidades
        return output

model = Net()
print(model)

Para conferir os shapes indicados nos comentários, passamos um mini-batch pelas camadas, uma a uma:

In [ ]:
with torch.no_grad():
    x = data[:4]                                    ; print('entrada      ', tuple(x.shape))
    x = torch.flatten(x, 1)                         ; print('flatten      ', tuple(x.shape))
    x = F.relu(model.fc1(x))                        ; print('fc1 + relu   ', tuple(x.shape))
    x = F.relu(model.fc2(x))                        ; print('fc2 + relu   ', tuple(x.shape))
    x = model.fc3(x)                                ; print('fc3 (logits) ', tuple(x.shape))

## Treinamento

### Função de custo

A rede devolve log-probabilidades, e a função de custo é `F.nll_loss` (*negative log-likelihood*): para cada amostra, pega a log-probabilidade da classe correta e troca o sinal; depois tira a média do mini-batch. `log_softmax` seguido de `nll_loss` é exatamente a cross-entropy dos notebooks 3.x. Seria equivalente tirar o `log_softmax` do `forward` e usar `nn.CrossEntropyLoss` (que já aplica o log-softmax) sobre os logits; o que não se deve fazer é aplicar softmax antes de `nn.CrossEntropyLoss`.

### Funções de treino e de teste

`train` faz uma época: para cada mini-batch, copia os dados para o `device` (`.to(device)`), calcula a saída e o custo, zera os gradientes, retropropaga e atualiza os parâmetros. `model.train()` liga o modo de treino, em que o dropout está ativo.

`test` avalia o modelo no conjunto de teste. `model.eval()` liga o modo de avaliação (o dropout é desligado) e `torch.no_grad()` desliga o registro das operações para o gradiente, que não é necessário aqui e gastaria memória. Com `reduction='sum'`, o custo de cada lote é somado e depois dividido pelo número total de imagens. A classe prevista é a de maior log-probabilidade (`argmax`).

In [ ]:
def train(model, device, train_loader, optimizer, epoch, log_interval=100):
    model.train()                       # modo de treino: dropout ativo
    losses = []
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if batch_idx % log_interval == 0:
            print('Época de treino: {} [{}/{} ({:.0f}%)]\tCusto: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
    return losses

In [ ]:
def test(model, device, test_loader):
    model.eval()                        # modo de avaliação: dropout desligado
    test_loss = 0
    correct = 0
    with torch.no_grad():               # sem gradientes na avaliação
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # soma o custo do lote
            pred = output.argmax(dim=1, keepdim=True)  # índice da maior log-probabilidade
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    print('\nTeste: custo médio: {:.4f}, acurácia: {}/{} ({:.2f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset), accuracy))
    return test_loss, accuracy

### Otimizador e agendador da taxa de aprendizado

- **Adadelta** (`optim.Adadelta`): um otimizador adaptativo da família do RMSprop (notebook 4.4). Além da média móvel dos gradientes ao quadrado, ele guarda uma média móvel dos quadrados das próprias atualizações e usa a razão entre as duas para definir o passo de cada parâmetro. No PyTorch, `lr` multiplica esse passo; `lr=1.0` é o valor padrão.
- **StepLR** (`StepLR(optimizer, step_size=1, gamma=0.7)`): um agendador (*scheduler*) que multiplica a taxa de aprendizado por `gamma` a cada `step_size` épocas. Aqui, na época $e$ (começando em 1), $\eta_e = 1{,}0 \times 0{,}7^{\,e-1}$. `scheduler.step()` é chamado uma vez por época, depois do treino.

Usamos as 14 épocas do exemplo oficial. Com a rede densa, o treino inteiro leva poucos minutos em CPU e menos de um minuto numa GPU. Acompanhe a taxa de aprendizado caindo a cada época: na 14ª, ela é $0{,}7^{13} pprox 0{,}01$, cem vezes menor que a inicial.

In [ ]:
model = Net().to(device)              # .to(device): parâmetros na GPU, se houver
optimizer = optim.Adadelta(model.parameters(), lr=1.0)
scheduler = StepLR(optimizer, step_size=1, gamma=0.7)

n_epochs = 14   # como no exemplo oficial do PyTorch

losses, test_losses, test_accs = [], [], []
t0 = time.time()
for epoch in range(1, n_epochs + 1):
    print(f'Época {epoch}: lr = {scheduler.get_last_lr()[0]:.3f}')
    losses += train(model, device, train_loader, optimizer, epoch)
    test_loss, test_acc = test(model, device, test_loader)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    scheduler.step()                  # atualiza a taxa de aprendizado para a próxima época
print(f'tempo de treinamento: {time.time() - t0:.0f} s')

In [ ]:
n_batches = len(train_loader)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot([i / n_batches for i in range(len(losses))], losses, lw=0.5, label='treino (mini-batch)')
ax1.plot(range(1, n_epochs + 1), test_losses, 'o-', label='teste (fim da época)')
ax1.set_xlabel('épocas'); ax1.set_ylabel('custo (NLL)'); ax1.set_yscale('log'); ax1.legend()
ax2.plot(range(1, n_epochs + 1), test_accs, 'o-')
ax2.set_xlabel('épocas'); ax2.set_ylabel('acurácia no teste (%)'); ax2.set_xticks(range(1, n_epochs + 1))
plt.show()

## Gravação e leitura do modelo

`model.state_dict()` devolve um dicionário com o nome e o tensor de cada parâmetro (os pesos e bias de cada camada). `torch.save` grava esse dicionário num arquivo. Para usar o modelo depois, criamos uma rede com a mesma arquitetura e carregamos os parâmetros com `load_state_dict`. O modelo carregado deve ter a mesma acurácia no teste.

In [ ]:
torch.save(model.state_dict(), "mnist_mlp.pt")
print({name: tuple(t.shape) for name, t in model.state_dict().items()})

model2 = Net().to(device)
model2.load_state_dict(torch.load("mnist_mlp.pt", map_location=device))
_ = test(model2, device, test_loader)

## Avaliação: exemplos do conjunto de teste

Predições do modelo para as 10 primeiras imagens do teste e, abaixo, 10 imagens em que ele erra. A acurácia resume o desempenho, mas os erros mostram que tipo de imagem confunde a rede.

In [ ]:
model.eval()
with torch.no_grad():
    images, labels = next(iter(test_loader))
    preds = model(images.to(device)).argmax(dim=1).cpu()   # .cpu(): de volta à CPU para o matplotlib

wrong = (preds != labels).nonzero().flatten()[:10]
fig, axs = plt.subplots(2, 10, figsize=(15, 3.8))
for ax, i in zip(axs[0], range(10)):
    ax.imshow(images[i, 0], cmap='gray')
    ax.set_title(f'prev. {preds[i].item()} / real {labels[i].item()}', fontsize=9)
    ax.axis('off')
for ax in axs[1]:
    ax.axis('off')
for ax, i in zip(axs[1], wrong):
    ax.imshow(images[i, 0], cmap='gray')
    ax.set_title(f'prev. {preds[i].item()} / real {labels[i].item()}', fontsize=9, color='r')
plt.show()
print(f'erros nas {len(labels)} primeiras imagens do teste: {(preds != labels).sum().item()}')

## Exercícios

1. Conte os parâmetros de cada camada da rede (use `model.named_parameters()` e `p.numel()`) e o total. Qual camada tem mais parâmetros? Confira a conta de `fc1` à mão: 784 × 512 pesos mais 512 bias.

In [ ]:
# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

2. A rede termina com `log_softmax` e o custo é `F.nll_loss`. Verifique numericamente, com os logits de um mini-batch (a saída de `fc2`), que `F.nll_loss(F.log_softmax(z, dim=1), target)` dá o mesmo valor que `F.cross_entropy(z, target)`. Calcule também o valor (errado) que se obtém aplicando `F.softmax` antes de `F.cross_entropy`.

In [ ]:
# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

3. O que acontece se esquecermos o `model.eval()` na função `test`? E se esquecermos o `torch.no_grad()`? Verifique avaliando o modelo treinado com o modo de treino ligado (`model.train()` antes do laço de teste).

In [ ]:
# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

4. Troque o Adadelta por Adam (`optim.Adam(model.parameters(), lr=1e-3)`), mantendo o `StepLR`, e treine uma rede nova pelo mesmo número de épocas. Antes de executar, preveja se a acurácia no teste será maior ou menor que a do Adadelta (`test_accs`). Compare também o custo no teste. Com uma só execução de cada, dá para concluir qual otimizador é melhor para este problema?

In [ ]:
# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

## Aprendizados

- O pipeline completo tem sempre as mesmas peças: `Dataset`/`DataLoader`, modelo (`nn.Module`), função de custo, otimizador, laço de treino por épocas e mini-batches, e avaliação num conjunto de teste separado.
- `model.train()` e `model.eval()` mudam o comportamento de camadas como o dropout; `torch.no_grad()` só desliga o cálculo de gradientes. Na avaliação usamos os dois.
- `.to(device)` coloca modelo e dados no mesmo dispositivo; o mesmo código roda em CPU e em GPU.
- Uma rede densa trata a imagem achatada como 784 números independentes; mesmo sem saber quais pixels são vizinhos, ela passa de 98% de acurácia no MNIST. A primeira camada concentra a maior parte dos parâmetros.
- `state_dict` guarda os parâmetros treinados; para reutilizá-los, cria-se a mesma arquitetura e chama-se `load_state_dict`.